# BSMM-8730 — Smart Centres REIT Analysis
## Notebook 1B: Data Collection — Bank of Canada Inflation (CPI)

**Author:** Muhammad Ahmad
**Role:** Member C — Bank of Canada Valet API
**Branch:** feature/bankofcanada-inflation
**Date:** July 2026

### What this notebook does:
- Connects to the Bank of Canada Valet API
- Downloads the Consumer Price Index (CPI) history (2020 to 2025)
- Saves the data as a CSV file
- Downloads the CSV to your computer

### What is CPI and why does it matter?
CPI = Consumer Price Index.
Think of it as a basket of everyday goods like groceries, gas, and rent.
When the CPI number goes up, those things cost more — that is inflation.
High inflation caused the Bank of Canada to raise interest rates in 2022-2023,
which directly hurt Smart Centres REIT stock price.

### Data Source:
- API: Bank of Canada Valet API
- URL: https://www.bankofcanada.ca/valet-api-how-to/
- Series V41690973 = Consumer Price Index (CPI)
- Frequency: Monthly (one reading per month)

---

## Step 0 — Install Libraries

In [ ]:
!pip install pandas requests --quiet
print("Done!")

## Step 1 — Import Libraries

In [ ]:
import requests
import pandas as pd
import os
from google.colab import files

print("All libraries imported!")

## Step 2 — Define Settings

In [ ]:
START_DATE        = "2020-01-01"
END_DATE          = "2025-12-31"
INFLATION_SERIES  = "V41690973"

INFLATION_URL = (
    f"https://www.bankofcanada.ca/valet/observations/{INFLATION_SERIES}/json"
    f"?start_date={START_DATE}&end_date={END_DATE}"
)

print(f"Series     : {INFLATION_SERIES}")
print(f"Date range : {START_DATE} to {END_DATE}")
print(f"API URL    : {INFLATION_URL}")
print()
print("Note: CPI data is monthly so expect around 60-70 rows total.")
print("That is completely normal — one reading per month.")

## Step 3 — Create raw_data Folder

In [ ]:
os.makedirs("raw_data", exist_ok=True)
print("Folder raw_data is ready!")

## Step 4 — Call the API

In [ ]:
print("Calling Bank of Canada API for inflation (CPI) data...")

response = requests.get(INFLATION_URL)

if response.status_code == 200:
    print(f"Success! Status code: {response.status_code}")
else:
    print(f"Error. Status code: {response.status_code}")
    print("Check your internet connection and try again.")

## Step 5 — Parse the Response

In [ ]:
data         = response.json()
observations = data["observations"]

print(f"Total observations received: {len(observations)}")
print()
print("Example raw observation:")
print(observations[0])

In [ ]:
# Extract date and CPI value from each observation
rows = []
for obs in observations:
    date  = obs["date"]
    value = obs.get(INFLATION_SERIES, {}).get("v", None)
    rows.append({"date": date, "cpi_value": value})

# Convert to DataFrame
df = pd.DataFrame(rows)

# Fix data types
df["date"]      = pd.to_datetime(df["date"])
df["cpi_value"] = pd.to_numeric(df["cpi_value"], errors="coerce")

print(f"Rows extracted: {len(df)}")
print()
print(df.head(10).to_string(index=False))

## Step 6 — Validate the Data

In [ ]:
print("=== Inflation (CPI) Data Summary ===")
print(f"  Date range    : {df['date'].min().date()} to {df['date'].max().date()}")
print(f"  Total rows    : {len(df)}")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"  Lowest CPI    : {df['cpi_value'].min()}")
print(f"  Highest CPI   : {df['cpi_value'].max()}")
print()
print("What the CPI number means:")
print("  CPI represents a basket of everyday goods priced at 100 in a base year.")
print("  A CPI of 150 means those goods now cost 50% more than the base year.")
print("  Rising CPI = rising inflation = pressure on Bank of Canada to raise rates.")

## Step 7 — Save as CSV

In [ ]:
FILENAME = "raw_data/bank_of_canada_inflation_cpi.csv"
df.to_csv(FILENAME, index=False)

print(f"Saved: {FILENAME}")
print(f"Rows : {len(df)}")

## Step 8 — Download to Your Computer

In [ ]:
files.download(FILENAME)

print("Downloaded!")
print()
print("Next steps:")
print("  1. Go to GitHub repo: 8730-assignment")
print("  2. Switch to branch : feature/bankofcanada-inflation")
print("  3. Upload this notebook and the CSV to raw_data/ folder")
print("  4. Commit message   : Add Bank of Canada inflation CPI data")
print("  5. Open PR 2")